In [1]:
# Install necessary libraries
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cpu
!pip install transformers librosa mamkit

Looking in indexes: https://download.pytorch.org/whl/cpu
  Using cached https://download.pytorch.org/whl/cpu/torch-2.1.2%2Bcpu-cp311-cp311-linux_x86_64.whl (184.9 MB)
  Using cached https://download.pytorch.org/whl/cpu/torchaudio-2.1.2%2Bcpu-cp311-cp311-linux_x86_64.whl (1.6 MB)
  Attempting uninstall: torch
    Found existing installation: torch 2.3.0
    Uninstalling torch-2.3.0:
      Successfully uninstalled torch-2.3.0
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.3.0
    Uninstalling torchaudio-2.3.0:
      Successfully uninstalled torchaudio-2.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mamkit 0.1.1 requires torch==2.3.0, but you have torch 2.1.2+cpu which is incompatible.
mamkit 0.1.1 requires torchaudio==2.3.0, but you have torchaudio 2.1.2+cpu which is incompatible.
torchtext 0.18.0 requires torch>=2.3.0, but

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from pathlib import Path
from mamkit.data.datasets import MMUSEDFallacy, InputMode
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import RobertaTokenizer
import torchaudio
import torch
import torch.nn.functional as F
import torch.nn as nn
from transformers import RobertaModel, WavLMModel


/usr/local/lib/python3.11/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.11/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [4]:
#  2. Load Dataset
DATA_PATH = Path("/content/base")
loader_b = MMUSEDFallacy(task_name="afc", input_mode=InputMode.TEXT_AUDIO, base_data_path=DATA_PATH)
splits_b = loader_b.get_splits()

from sklearn.model_selection import train_test_split
train_val_split, _ = train_test_split(splits_b[0].train, test_size=0.2, random_state=42)
train_split, val_split = train_test_split(train_val_split, test_size=0.2, random_state=42)

Downloading: https://zenodo.org/api/records/14938117/files-archive: 5.71MB [00:01, 3.41MB/s]
  0%|          | 0/35 [00:00<?, ?it/s]

[youtube] Extracting URL: https://www.youtube.com/watch?v=_hEeMvhcNlE&t=4579s
[youtube] _hEeMvhcNlE: Downloading webpage
[youtube] _hEeMvhcNlE: Downloading tv client config
[youtube] _hEeMvhcNlE: Downloading player d71837c8
[youtube] _hEeMvhcNlE: Downloading tv player API JSON
[youtube] _hEeMvhcNlE: Downloading ios player API JSON


         player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         n = DpcXWHoPYxD0v92g ; player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         n = M60ggIjzhXpM4ERF ; player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         n = fUMQ_EEEfNi66eO7 ; player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js


[youtube] _hEeMvhcNlE: Downloading m3u8 information
[info] _hEeMvhcNlE: Downloading 1 format(s): 234
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1094
[download] Destination: /content/base/MMUSED-fallacy/audio_recordings/30_2004/full_audio
[download] 100% of   90.82MiB in 00:02:24 at 645.64KiB/s                   
[ExtractAudio] Destination: /content/base/MMUSED-fallacy/audio_recordings/30_2004/full_audio.wav
Deleting original file /content/base/MMUSED-fallacy/audio_recordings/30_2004/full_audio (pass -k to keep)


  3%|▎         | 1/35 [02:39<1:30:20, 159.42s/it]

[youtube] Extracting URL: https://www.youtube.com/watch?v=xdQK78l3Xhc&t=119s
[youtube] xdQK78l3Xhc: Downloading webpage
[youtube] xdQK78l3Xhc: Downloading tv client config
[youtube] xdQK78l3Xhc: Downloading player d71837c8
[youtube] xdQK78l3Xhc: Downloading tv player API JSON
[youtube] xdQK78l3Xhc: Downloading ios player API JSON


... (131 outputs left out)

[youtube] Extracting URL: https://www.youtube.com/watch?v=UCA1A5GqCdQ&ab_channel=NBCNews
[youtube] UCA1A5GqCdQ: Downloading webpage
[youtube] UCA1A5GqCdQ: Downloading tv client config
[youtube] UCA1A5GqCdQ: Downloading player d71837c8
[youtube] UCA1A5GqCdQ: Downloading tv player API JSON
[youtube] UCA1A5GqCdQ: Downloading ios player API JSON


         player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         n = YXYMwP_bxceePvqh ; player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         n = 1Eej-_Uhy-W7ialv ; player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js
         n = Ba_0wb74MGHPjpOY ; player = https://www.youtube.com/s/player/d71837c8/player_ias.vflset/en_US/base.js


[youtube] UCA1A5GqCdQ: Downloading m3u8 information
[info] UCA1A5GqCdQ: Downloading 1 format(s): 234
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 1345
[download] Destination: /content/base/MMUSED-fallacy/audio_recordings/46_2020/full_audio
[download] 100% of  111.60MiB in 00:00:41 at 2.66MiB/s                     
[ExtractAudio] Destination: /content/base/MMUSED-fallacy/audio_recordings/46_2020/full_audio.wav
Deleting original file /content/base/MMUSED-fallacy/audio_recordings/46_2020/full_audio (pass -k to keep)


Building AFC Context: 100%|██████████| 1278/1278 [00:00<00:00, 11096.46it/s]


In [5]:
from collections import Counter

class MultimodalDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128, audio_max_len=16000):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.audio_max_len = audio_max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, audio_path_list, label, *_ = self.data[idx]
        audio_path = audio_path_list[0] if isinstance(audio_path_list, list) else audio_path_list

        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors='pt')

        waveform, sample_rate = torchaudio.load(audio_path)
        if sample_rate != 16000:
            waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=16000)(waveform)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        waveform = waveform.squeeze()
        waveform = F.pad(waveform, (0, max(0, self.audio_max_len - waveform.size(0))))[:self.audio_max_len]

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'audio_input': waveform,
            'label': torch.tensor(label)
        }


tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
train_dataset = MultimodalDataset(train_split, tokenizer)
val_dataset = MultimodalDataset(val_split, tokenizer)

train_labels = [label for _, _, label, *_ in train_split]
label_counts = Counter(train_labels)
weights = [1.0 / label_counts[label] for label in train_labels]
sampler = WeightedRandomSampler(weights, len(weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [6]:
# Get all labels from the training split
labels = [label for _, _, label, *_ in splits_b[0].train]

# Get the number of unique labels (i.e., number of classes)
num_classes = len(set(labels))
print(f"Number of classes: {num_classes}")


Number of classes: 6


In [7]:
from collections import Counter

label_counts = Counter(labels)
print("Label distribution:")
for label, count in sorted(label_counts.items()):
    print(f"Class {label}: {count} samples")


Label distribution:
Class 0: 800 samples
Class 1: 191 samples
Class 2: 149 samples
Class 3: 56 samples
Class 4: 46 samples
Class 5: 36 samples


In [8]:
#  4. Model Definition


num_fallacy_classes = 6

class MultimodalClassifier(nn.Module):
    def __init__(self, num_labels):
        super(MultimodalClassifier, self).__init__()
        self.text_encoder = RobertaModel.from_pretrained("/content/drive/MyDrive/models/roberta-base")
        self.audio_encoder = WavLMModel.from_pretrained("/content/drive/MyDrive/models/wavlm-base-plus")
        self.dropout = nn.Dropout(0.3)
        self.hidden = nn.Linear(self.text_encoder.config.hidden_size + self.audio_encoder.config.hidden_size, 256)
        self.norm = nn.LayerNorm(256)
        self.classifier = nn.Linear(256, num_labels)

    def forward(self, input_ids, attention_mask, audio_input):
        text_outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        audio_outputs = self.audio_encoder(audio_input)

        text_pooled = text_outputs.pooler_output
        audio_pooled = audio_outputs.last_hidden_state.mean(dim=1)

        combined = torch.cat((text_pooled, audio_pooled), dim=1)
        hidden = self.norm(self.hidden(self.dropout(combined)))
        logits = self.classifier(hidden)
        return logits


In [9]:
from transformers import RobertaModel, WavLMModel

# Save RoBERTa and WavLM models to Drive (run this only once!)
roberta = RobertaModel.from_pretrained("roberta-base")
roberta.save_pretrained("/content/drive/MyDrive/models/roberta-base")

wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
wavlm.save_pretrained("/content/drive/MyDrive/models/wavlm-base-plus")


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

In [10]:
# 5. Training

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalClassifier(num_labels=num_fallacy_classes).to(device)

# Class-weighted loss
total = sum(label_counts.values())
class_weights = torch.tensor([total / label_counts[i] for i in range(num_fallacy_classes)], dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

from sklearn.metrics import classification_report

for epoch in range(7):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        audio_input = batch['audio_input'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask, audio_input)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1}, Train Loss: {total_loss / len(train_loader):.4f}")

    # Validation
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            audio_input = batch['audio_input'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask, audio_input)
            preds = torch.argmax(outputs, dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    print("Validation Classification Report:")
    print(classification_report(val_labels, val_preds, digits=4))


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/conv.py:306: UserWarning: Plan failed with a cudnnException: CUDNN_BACKEND_EXECUTION_PLAN_DESCRIPTOR: cudnnFinalize Descriptor Failed cudnn_status: CUDNN_STATUS_NOT_SUPPORTED (Triggered internally at ../aten/src/ATen/native/cudnn/Conv_v8.cpp:919.)
  return F.conv1d(input, weight, bias, self.stride,


Epoch 1, Train Loss: 1.0629
Validation Classification Report:
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       136
           1     0.1667    0.0400    0.0645        25
           2     0.1792    0.7308    0.2879        26
           3     0.1500    0.5000    0.2308         6
           4     0.0727    0.6667    0.1311         6
           5     0.1111    0.3333    0.1667         6

    accuracy                         0.1415       205
   macro avg     0.1133    0.3785    0.1468       205
weighted avg     0.0528    0.1415    0.0599       205



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2, Train Loss: 0.3569
Validation Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.0074    0.0146       136
           1     0.2464    0.6800    0.3617        25
           2     0.4167    0.7692    0.5405        26
           3     0.0678    0.6667    0.1231         6
           4     0.0000    0.0000    0.0000         6
           5     0.2143    0.5000    0.3000         6

    accuracy                         0.2195       205
   macro avg     0.3242    0.4372    0.2233       205
weighted avg     0.7546    0.2195    0.1347       205

Epoch 3, Train Loss: 0.1548
Validation Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.0368    0.0709       136
           1     0.2091    0.9200    0.3407        25
           2     0.4474    0.6538    0.5312        26
           3     0.1600    0.6667    0.2581         6
           4     0.0769    0.1667    0.1053         6
        

In [11]:
torch.save(model.state_dict(), "/content/drive/MyDrive/task_b_model.pt")